# 1.4 Preprocessed Data — Feature Engineering

Transform the preprocessed data into model-ready features.

**Steps:**
1. Load preprocessed data
2. Encode categorical variables
3. Create new features
4. Scale numerical features
5. Final feature set validation
6. Save engineered features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1.4.1 Load Preprocessed Data (Train + Test)

In [ ]:
df_train = pd.read_csv('../data/processed/house_data_preprocessed_train.csv')
df_test = pd.read_csv('../data/processed/house_data_preprocessed_test.csv')

print(f'Loaded {df_train.shape[0]} train rows and {df_train.shape[1]} columns')
print(f'Loaded {df_test.shape[0]} test rows and {df_test.shape[1]} columns')
df_train.head()

## 1.4.2 Encode Categorical Variables

In [ ]:
LOCATION_CATEGORIES = ['Downtown', 'Mountain', 'Rural', 'Suburb', 'Urban', 'Waterfront']

df_train_encoded = df_train.copy()
df_train_encoded['location'] = pd.Categorical(df_train['location'], categories=LOCATION_CATEGORIES)
df_test_encoded = df_test.copy()
df_test_encoded['location'] = pd.Categorical(df_test['location'], categories=LOCATION_CATEGORIES)

df_train_encoded = pd.get_dummies(df_train_encoded, columns=['location'], prefix='loc', drop_first=False)
df_test_encoded = pd.get_dummies(df_test_encoded, columns=['location'], prefix='loc', drop_first=False)

location_dummies = [c for c in df_train_encoded.columns if c.startswith('loc_')]
df_test_encoded = df_test_encoded.reindex(columns=df_train_encoded.columns, fill_value=0)

print(f'After one-hot encoding location: {df_train_encoded.shape[1]} columns')
print(f'Location dummies: {location_dummies}')

In [ ]:
condition_order = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}
df_train_encoded['condition_encoded'] = df_train_encoded['condition'].map(condition_order)
df_test_encoded['condition_encoded'] = df_test_encoded['condition'].map(condition_order)

print('Condition encoding:')
for k, v in condition_order.items():
    print(f'  {k} -> {v}')

df_train_encoded.drop(columns=['condition'], inplace=True)
df_test_encoded.drop(columns=['condition'], inplace=True)
print(f'\nFinal train columns: {df_train_encoded.columns.tolist()}')

## 1.4.3 Create New Features

In [ ]:
df_train_encoded['total_rooms'] = df_train_encoded['bedrooms'] + df_train_encoded['bathrooms']
df_train_encoded['bath_bed_ratio'] = df_train_encoded['bathrooms'] / df_train_encoded['bedrooms'].replace(0, 1)
df_train_encoded['is_luxury'] = ((df_train_encoded['condition_encoded'] >= 3) & (df_train_encoded['sqft'] >= 2500)).astype(int)
df_train_encoded['is_new'] = (df_train_encoded['house_age'] <= 15).astype(int)
df_train_encoded['sqft_per_bedroom'] = df_train_encoded['sqft'] / df_train_encoded['bedrooms'].replace(0, 1)
df_train_encoded['log_price'] = np.log1p(df_train_encoded['price'])

df_test_encoded['total_rooms'] = df_test_encoded['bedrooms'] + df_test_encoded['bathrooms']
df_test_encoded['bath_bed_ratio'] = df_test_encoded['bathrooms'] / df_test_encoded['bedrooms'].replace(0, 1)
df_test_encoded['is_luxury'] = ((df_test_encoded['condition_encoded'] >= 3) & (df_test_encoded['sqft'] >= 2500)).astype(int)
df_test_encoded['is_new'] = (df_test_encoded['house_age'] <= 15).astype(int)
df_test_encoded['sqft_per_bedroom'] = df_test_encoded['sqft'] / df_test_encoded['bedrooms'].replace(0, 1)
df_test_encoded['log_price'] = np.log1p(df_test_encoded['price'])

print('Created 6 new features (train and test)')
df_train_encoded[['price', 'log_price', 'total_rooms', 'bath_bed_ratio', 'is_luxury', 'is_new', 'sqft_per_bedroom']].head(10)

## 1.4.4 Scale Numerical Features (scaler fitted on train)

In [ ]:
features_to_scale = ['sqft', 'bedrooms', 'bathrooms', 'house_age',
                     'total_rooms', 'bath_bed_ratio', 'sqft_per_bedroom']

scaler = StandardScaler()
df_train_scaled = df_train_encoded.copy()
df_train_scaled[features_to_scale] = scaler.fit_transform(df_train_encoded[features_to_scale])

df_test_scaled = df_test_encoded.copy()
df_test_scaled[features_to_scale] = scaler.transform(df_test_encoded[features_to_scale])

print('Scaler fitted on TRAIN split only; test scaled with the same scaler.')
print('Scaled features statistics (train):')
df_train_scaled[features_to_scale].describe().round(2)

## 1.4.5 Feature Importance Preview

In [ ]:
feature_cols = [c for c in df_train_scaled.columns if c not in ['price', 'log_price']]
corr_with_target = df_train_scaled[feature_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in corr_with_target.values]
corr_with_target.plot(kind='barh', ax=ax, color=colors, edgecolor='black', alpha=0.7)
ax.set_title('Feature Correlation with Log Price (train)')
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('\nCorrelation with log_price:')
print(corr_with_target.round(3))

## 1.4.6 Final Feature Set

In [ ]:
FEATURE_COLS = [c for c in df_train_scaled.columns
                if c not in ['price', 'log_price']]

df_test_scaled = df_test_scaled.reindex(columns=df_train_scaled.columns, fill_value=0)

print(f'Final feature count: {len(FEATURE_COLS)}')
print(f'\nFeatures:')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

print(f'\nTarget: price (log_price for training)')
print(f'\nTrain feature matrix shape: {df_train_scaled[FEATURE_COLS].shape}')
print(f'Test feature matrix shape:  {df_test_scaled[FEATURE_COLS].shape}')

## 1.4.7 Save Feature-Ready Data

In [ ]:
features_dir = '../data/featured'
os.makedirs(features_dir, exist_ok=True)

train_output_path = os.path.join(features_dir, 'house_data_features_train.csv')
test_output_path = os.path.join(features_dir, 'house_data_features_test.csv')
df_train_scaled.to_csv(train_output_path, index=False)
df_test_scaled.to_csv(test_output_path, index=False)
print(f'Saved train feature dataset to {train_output_path}')
print(f'Train shape: {df_train_scaled.shape}')
print(f'Saved test feature dataset to {test_output_path}')
print(f'Test shape: {df_test_scaled.shape}')

feature_list_path = os.path.join(features_dir, 'feature_list.json')
with open(feature_list_path, 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f'Saved feature list to {feature_list_path}')

scaler_params = {
    'features': features_to_scale,
    'mean': dict(zip(features_to_scale, scaler.mean_)),
    'std': dict(zip(features_to_scale, scaler.scale_))
}
scaler_path = os.path.join(features_dir, 'scaler_params.json')
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f, indent=2)
print(f'Saved scaler parameters to {scaler_path}')